In [20]:
#Import Libraries, scikit Learn for the loading of the data set
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np
import matplotlib as plt

In [21]:
# Fetch dataset directly as a Pandas DataFrame
mnist = fetch_openml('mnist_784', version=1, as_frame=True)

# Combine feature columns and labels into a single DataFrame
df = mnist.data.copy()
df['label'] = mnist.target
df["label"] = df["label"].astype(int)
df.head

<bound method NDFrame.head of        pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  pixel8  pixel9  \
0           0       0       0       0       0       0       0       0       0   
1           0       0       0       0       0       0       0       0       0   
2           0       0       0       0       0       0       0       0       0   
3           0       0       0       0       0       0       0       0       0   
4           0       0       0       0       0       0       0       0       0   
...       ...     ...     ...     ...     ...     ...     ...     ...     ...   
69995       0       0       0       0       0       0       0       0       0   
69996       0       0       0       0       0       0       0       0       0   
69997       0       0       0       0       0       0       0       0       0   
69998       0       0       0       0       0       0       0       0       0   
69999       0       0       0       0       0       0       0       0       0  

In [22]:
def init_data(data_frame , split_size):
    data = np.array(df)
    n , m = data.shape
    
    #Shuffle data for the randomization
    np.random.shuffle(data)
    
    #Split Ratio 
    split = int(split_size * n)
    
    #Split data
    train = data[:split]
    test = data[split:]
    
    #Create X , Y from splits
    x_train = train[:, :-1]
    y_train = train[:, -1]
    
    x_test = test[:, :-1]
    y_test = test[:, -1]
    return x_train , y_train , x_test , y_test

In [23]:
#Functions
def init_weights():
    w1 = np.random.uniform(-1, 1, (10, 784))
    w2 = np.random.uniform(-1, 1, (10, 10))
    
    b1 = np.random.uniform(-1, 1, (10, 1))
    b2 = np.random.uniform(-1, 1, (10, 1))

    return w1 , b1 , w2 , b2

def relu(s):
    return np.maximum(0 , s)

def softmax(s):
    s = s - np.max(s, axis=0, keepdims=True)
    return np.exp(s) / np.sum(np.exp(s), axis=0, keepdims=True)

def onehot(y):
    y_train_len = len(y)
    y_train_onehot = np.zeros((10, y_train_len))
    y_train_onehot[y_train.astype(int), np.arange(y_train_len)] = 1
    return y_train_onehot

def d_relu(s):
    return s > 0
    
def forward(x , w1 , b1 , w2 , b2):
    a0 = x.T
    s1 = w1.dot(a0) + b1
    a1 = relu(s1)

    s2 = w2.dot(a1) + b2
    a2 = softmax(s2)
    
    return a1 , a2

def backwards(x , y  , a1 , a2 , w2):
    onehot_y = onehot(y)
    m = len(y)
    gradient1 = a2 - onehot_y

    dw2 = 1/m * gradient1.dot(a2.T)
    db2 = 1/m * np.sum(gradient1, axis=1, keepdims=True)

    gradient2 = w2.T.dot(gradient1) * d_relu(a1)
    dw1 = 1/m * gradient2.dot(x) 
    db1 = 1/m * np.sum(gradient2, axis=1, keepdims=True)

    return dw1 , db1 , dw2 , db2

def update_weights(w1, b1, w2, b2, dw1, db1, dw2, db2, alpha):
    w1 = w1 - alpha*dw1
    w2 = w2 - alpha*dw2

    b1 = b1 - alpha*db1
    b2 = b2 - alpha*db2
    return w1 , b1 , w2 , b2



In [24]:
def learning_loop(x_train , y_train , itterations , learning_rate):
    w1 , b1 , w2 , b2 = init_weights()
    alpha  = learning_rate
    for i in range(itterations):
        a1 , a2 = forward(x_train , w1 , b1 , w2 , b2)
        dw1 , db1 , dw2 , db2 = backwards(x_train , y_train  , a1 , a2 , w2)
        w1 , b1 , w2 , b2 = update_weights(w1, b1, w2, b2, dw1, db1, dw2, db2, alpha)
    return w1 , b1 , w2 , b2

def network_check(x , y , w1 , b1 , w2 , b2):
    a0 = x.T
    s1 = w1.dot(a0) + b1
    a1 = relu(s1)

    s2 = w2.dot(a1) + b2
    a2 = softmax(s2)
    
    yhat = np.argmax(a2 , axis = 0)

    accuracy = np.sum(yhat == y)/ y.size
    return accuracy

In [25]:
x_train , y_train , x_text , y_test = init_data(df , 0.8)
learning_rate = 0.1
itterations = 20

w1 , b1 , w2 , b2 = learning_loop(x_train , y_train , itterations , learning_rate)

In [26]:
accuracy = network_check(x_text , y_test , w1 , b1 , w2 , b2)
print(f"Accuracy of the Modle is : {accuracy}")

Accuracy of the Modle is : 0.09778571428571428
